#### **LIMPIEZA DE DATOS**

In [13]:
import pandas as pd 
import numpy as np
import re

catalogo_productos = pd.read_csv("Datos-originales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-originales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-originales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-originales/procurement_cajas.csv")

In [14]:
catalogo_productos.head()

,codigo_producto,descripcion_producto,ingrediente_forma,tipo_proyecto,subcategoria,categoria,tamaño_corte,peso_neto_caja,tamaño_paquete,cantidad_paquetes,peso_neto_paquete,caja_tipo_id
0,BR0001,"5 bolsas de 2,5 kg | Bastones rectos de brocol...",Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Fino,12.50,"5 X 2,5 KG",5.0,2.50,a23eeef1c0cd8cfd40b780d65382034a
1,BR0002,Espirales de brocoli - clasico - Caja de 13 pa...,Espirales de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,9.75,"13 X 0,75 KG",13.0,0.75,3256a5fed07b421a92abee44a3d93243
2,BR0003,Gajos de brocoli - sazonado - Caja de 13 packs...,Gajos de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,9.75,"13 X 0,75 KG",13.0,0.75,4f08f853cf7f9be28571df66a50bc322
3,BR0004,Bastones rectos de brocoli - crocante - Caja d...,Bastones rectos de brocoli,Estacional,Bastones clasicos - Sigilo,Bastones clasicos,Grueso,11.25,"9 X 1,25 KG",9.0,1.25,c96df5913565ba35f3e02e744c9a6dde
4,BR0005,"13 bolsas de 0,75 kg | Espirales de brocoli si...",Espirales de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,9.75,"13 X 0,75 KG",13.0,0.75,eb4895380ccaa410da0fefb6f12b197b


Observemos que hay incongruencias entre los valores predeterminados del peso neto de la caja, y las descripciones propias de cada producto. Vamos a priorizar la descripción dada, de manera que haría falta corregir la información de los datos.

In [15]:
def extraer_cantidad_peso(descripcion):
    """
    Extrae cantidad de paquetes y peso por paquete de la descripción.
    Retorna: (cantidad, peso_kg)
    """
    descripcion = str(descripcion).strip()
    
    # PATRÓN 1: "13 packs de 5 kg" o "13 bolsas de 2 kg"
    patron = r'(\d+)\s*(?:packs?|bolsas?|paquetes?)\s+de\s+([\d,]+\.?\d*)\s*(?:kg|KG)'
    match = re.search(patron, descripcion, re.IGNORECASE)
    if match:
        return int(match.group(1)), float(match.group(2).replace(',', '.'))
    
    # PATRÓN 2: "5 X 2,5 KG" o "2 x 2.5 kg"
    patron = r'(\d+)\s*[xX*]\s*([\d,]+\.?\d*)\s*(?:kg|KG)'
    match = re.search(patron, descripcion, re.IGNORECASE)
    if match:
        return int(match.group(1)), float(match.group(2).replace(',', '.'))

cantidades = []
pesos = []

for desc in catalogo_productos['descripcion_producto']:
    cant, peso = extraer_cantidad_peso(desc)
    cantidades.append(cant)
    pesos.append(peso)

# Asignar al DataFrame
catalogo_productos['cantidad_paquetes'] = cantidades
catalogo_productos['peso_neto_paquete'] = pesos

# Recalculamos los valores de 'peso_neto_caja'
catalogo_productos['peso_neto_caja'] = catalogo_productos['cantidad_paquetes'] * catalogo_productos['peso_neto_paquete']

In [16]:
especificaciones_cajas.head()

,caja_tipo_id,caja_grosor_mm,caja_interior_largo,caja_interior_ancho,caja_interior_alto,caja_exterior_largo,caja_exterior_ancho,caja_exterior_alto,pallet_alto,pallet_ancho,pallet_largo,cantidad_cajas_alto,cantidad_cajas_largo,cantidad_cajas_ancho,cantidad_cajas_total,utilizacion
0,02cf77de65b70dd77905e2e33d78478f,2.7,395.0,296.0,291.0,400.0,301.4,296.4,1800,800,1200,6.0,2.0,3.0,36.0,0.744458
1,082c1cdb42b1abd201403ca33ca11ef0,4.1,383.0,248.0,189.0,391.2,256.2,197.2,1800,800,1200,9.0,2.0,4.0,72.0,0.823519
2,0835ff365412a67b720a19713ec250f3,4.1,386.0,286.0,278.0,394.2,294.2,286.2,1800,800,1200,6.0,2.0,4.0,48.0,0.921990
3,0b72571a5bb7429ce7de424547e8d27d,4.1mm,386.0,286.0,174.0,394.2,294.2,182.2,1800,800,1200,9.0,2.0,4.0,72.0,0.880433
4,10c5f9edbe2c87186bcdeb991fe8d902,4.6,380.0,252.0,228.0,389.2,261.2,237.2,1800,800,1200,7.0,2.0,4.0,56.0,0.781457


Algunas medidas en caja_grosor_mm aparecen con la unidad 'mm', aprovechemos para además transformar la columna en float

In [17]:
especificaciones_cajas['caja_grosor_mm'] = (
    especificaciones_cajas['caja_grosor_mm']
    .astype(str)
    .str.replace('mm', '', regex=False)
    .str.strip()
    .str.replace(',', '.')
    .astype(float)
)

Faltan algunos cálculos de la cantidad_cajas_total

In [18]:
especificaciones_cajas['cantidad_cajas_total'] = (
    especificaciones_cajas['cantidad_cajas_alto'] * 
    especificaciones_cajas['cantidad_cajas_largo'] * 
    especificaciones_cajas['cantidad_cajas_ancho']
)

In [19]:
operaciones_planta.head()

,codigo_producto,volumen_producto_canal_servicios_comida,volumen_producto_canal_minorista,volumen_producto_canal_cadenas_corporativas,volumen_producto_canal_otros,volumen_producto_total,volumen_producto_planta_buenos_aires,volumen_producto_planta_curitiba,volumen_producto_planta_santiago,volumen_producto_planta_monterrey,...,costo_total_planta_santiago,costo_total_planta_monterrey,costo_total_planta_bakersfield,costo_pallets_planta_buenos_aires,costo_pallets_planta_curitiba,costo_pallets_planta_santiago,costo_pallets_planta_monterrey,costo_pallets_planta_bakersfield,costo_pallets_total,costo_total
0,BR0001,1537892,0,0,8721,1546613,0,1431746,26583,88284,...,17278.95,51646.14,0.00,0,5369100,99750,331200,0,5800050,720369.52
1,BR0002,0,139211,0,0,139211,0,139211,0,0,...,0.00,0.00,0.00,0,580050,0,0,0,580050,66821.28
2,BR0003,0,172506,0,0,172506,0,0,0,0,...,0.00,0.00,89703.12,0,0,0,0,323550,323550,89703.12
3,BR0004,0,271715,0,0,271715,0,0,0,0,...,0.00,0.00,130423.20,0,0,0,0,970500,970500,130423.20
4,BR0005,0,7586,0,0,7586,0,7586,0,0,...,0.00,0.00,0.00,0,23850,0,0,0,23850,3944.72


Nada que hacer en ese csv. Están todos los datos perfectos!

In [20]:
procurement_cajas.head()

,caja_tipo_id,volumen_tipo_planta_buenos_aires,volumen_tipo_planta_curitiba,volumen_tipo_planta_santiago,volumen_tipo_planta_monterrey,volumen_tipo_planta_bakersfield,costo_unitario_base,costo_total_base,costo_unitario_planta_buenos_aires,costo_unitario_planta_curitiba,costo_unitario_planta_santiago,costo_unitario_planta_monterrey,costo_unitario_planta_bakersfield,descuento_planta_buenos_aires,descuento_planta_curitiba,descuento_planta_santiago,descuento_planta_monterrey,descuento_planta_bakersfield,costo_por_pallet
0,02cf77de65b70dd77905e2e33d78478f,0,324352,1773376,0,2564276,0.60,504089.4,0.66,0.54,0.48,0.66,0.42,+10%,-10%,-20%,+10%,-30%,150
1,082c1cdb42b1abd201403ca33ca11ef0,0,617012,0,0,118368,0.65,238998.5,0.715,0.52,0.715,0.715,0.585,+10%,"-20,00%",+10%,+10%,-10%,150
2,0835ff365412a67b720a19713ec250f3,0,0,0,0,9196,0.65,5977.4,0.715,0.715,0.715,0.715,0.715,+10%,+10%,+10%,+10%,+10%,150
3,0b72571a5bb7429ce7de424547e8d27d,0,2144,71772,0,32428,0.65,34561.8,0.715,0.715,ERROR,0.715,0.715,+10%,+10%,+0 %,+10%,+10%,150
4,10c5f9edbe2c87186bcdeb991fe8d902,424662,480,312,162,186,0.70,49676.9,ERROR,0.77,0.77,0.77,0.77,-10,10%,+10%,+10%,+10%,150


Las columnas de descuentos están en formato string y puede ser difícil operar de esa forma. Recalculemos los valores dependiendo de los volúmenes anuales de cada tipo de caja.

- Tier 1 (< 20.000 unidades): precio base + 10% (markup) 
- Tier 2 (20.000–49.999 unidades): precio base (sin descuento) 
- Tier 3 (50.000–99.999 unidades): descuento del 10% 
- Tier 4 (100.000–499.999 unidades): descuento del 20 % 
- Tier 5 (≥ 500.000 unidades): descuento del 30 % (máximo)

In [21]:
# Calculemos primero cuántos tipos de caja se requieren para los productos
prod_op_merge = pd.concat([catalogo_productos, operaciones_planta], axis=1)

plantas = ['buenos_aires', 'curitiba', 'santiago', 'monterrey', 'bakersfield']
cols = [f'volumen_producto_planta_{p}' for p in plantas]

# Agrupamos los volúmenes de productos según cada planta por tipo de caja
volumen_por_caja = (prod_op_merge.groupby('caja_tipo_id')[cols].sum().reset_index())

# Renombramos las columnas para mejor control
volumen_por_caja.columns = ['caja_tipo_id'] + [f'volumen_tipo_planta_{p}_requerido' for p in plantas]
procurement_cajas = procurement_cajas.merge(volumen_por_caja, on='caja_tipo_id')

Ahora recalculamos los descuentos en base a las unidades anuales producidas.

In [22]:
volumen_cols = [c for c in procurement_cajas.columns if c.startswith('volumen_tipo_planta_') and c.endswith('requerido')]
descuento_cols = [col for col in procurement_cajas.columns if 'descuento_planta' in col]

def calcular_descuento_por_volumen(volumen):
    if volumen < 20000:
        return 0.10  # +10% markup
    elif volumen < 50000:
        return 0.00  # sin descuento
    elif volumen < 100000:
        return -0.10  # -10% descuento
    elif volumen < 500000:
        return -0.20  # -20% descuento
    else:
        return -0.30  # -30% descuento (máximo)

# Sobrescribir los descuentos
for col_vol in volumen_cols:
    planta = col_vol.replace('volumen_tipo_planta_', '').replace('_requerido', '')
    desc_col = f'descuento_planta_{planta}'
    procurement_cajas[desc_col] = procurement_cajas[col_vol].apply(calcular_descuento_por_volumen)

Ahora sí, aplicamos el descuento de cada planta al costo unitario base para recalcular el costo unitario por planta.

In [23]:
costo_base = pd.to_numeric(procurement_cajas['costo_unitario_base'], errors='coerce').fillna(0)

costo_cols = [c for c in procurement_cajas.columns if c.startswith('costo_unitario_planta_')]

for c in costo_cols:
    plant = c.replace('costo_unitario_planta_', '')
    desc_col = f'descuento_planta_{plant}'
    descuento = pd.to_numeric(procurement_cajas[desc_col], errors='coerce').fillna(0)
    procurement_cajas[c] = costo_base * (1 + descuento)

Exportamos las tablas modificadas en la carpeta "Datos-finales"

In [24]:
catalogo_productos.to_csv("Datos-finales/catalogo_productos.csv", index=False)
especificaciones_cajas.to_csv("Datos-finales/especificaciones_cajas.csv", index=False)
operaciones_planta.to_csv("Datos-finales/operaciones_planta.csv", index=False)
procurement_cajas.to_csv("Datos-finales/procurement_cajas.csv", index=False)